In [1]:
!pip install -q mediapipe opencv-python gradio gtts transformers torch  

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.15.3 which is incompatible.
mkl-umath 0.1.1 requires numpy<1.27.0,>=1.26.4, but you have numpy 2.2.6 which is incompatible.
mkl-random 1.2.4 requires numpy<1.27.0,>=1.26.4, but you have numpy 2.2.6 which is incompatible.
mkl-fft 1.3.8 requires numpy<1.27.0,>=1.26.4, but you have numpy 2.2.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
datasets 4.1.1 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is in

In [2]:
# 1. Wipe out the potentially corrupted versions
!pip uninstall -q -y mediapipe protobuf

# 2. Install specific versions known to work with MediaPipe 0.10.x
!pip install -q mediapipe==0.10.11 protobuf==3.20.3

# 3. Finalize with a compatible NumPy (to prevent that earlier error from returning)
!pip install -q "numpy<2.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
google-cloud-bigtable 2.32.0 requires google-api-core[grpc]<3.0.0,>=2.17.0, but you have google-api-core 1.34.1 which is incompatible.
bigframes 2.12.0 requires google-cloud-bigquery[bqstorage,pandas]>=3.31.0, but you have google-cloud-bigquery 3.25.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.1.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.1 which is incompatible.
pydrive2 1.21.3 requires pyOpenSSL<=24.2.

In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
import mediapipe as mp
import cv2
import os
import json
from gtts import gTTS
import gradio as gr
from transformers import pipeline

# Suppress Warnings
import warnings
warnings.filterwarnings('ignore')

print(f'Tensorflow V{tf.__version__}')
print(f'Keras V{tf.keras.__version__}')

2026-02-13 03:31:12.197941: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770953472.221315     358 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770953472.228243     358 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Tensorflow V2.18.0
Keras V3.8.0


In [4]:
INPUT_SIZE = 64
N_ROWS = 543
N_DIMS = 3
N_COLS = 0 # Will be updated after IDXS definition
NUM_CLASSES = 250
ROWS_PER_FRAME = 543

# Model Hyperparameters
LIPS_UNITS = 384
HANDS_UNITS = 384
POSE_UNITS = 384
UNITS = 512
NUM_BLOCKS = 2
MLP_RATIO = 2
EMBEDDING_DROPOUT = 0.00
MLP_DROPOUT_RATIO = 0.30
CLASSIFIER_DROPOUT_RATIO = 0.10

# Initializers and Activations
INIT_HE_UNIFORM = tf.keras.initializers.he_uniform
INIT_GLOROT_UNIFORM = tf.keras.initializers.glorot_uniform
INIT_ZEROS = tf.keras.initializers.constant(0.0)
GELU = tf.keras.activations.gelu

In [5]:
USE_TYPES = ['left_hand', 'pose', 'right_hand']
START_IDX = 468
LIPS_IDXS0 = np.array([
        61, 185, 40, 39, 37, 0, 267, 269, 270, 409,
        291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
        78, 191, 80, 81, 82, 13, 312, 311, 310, 415,
        95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
    ])
# Landmark indices in original data
LEFT_HAND_IDXS0 = np.arange(468,489)
RIGHT_HAND_IDXS0 = np.arange(522,543)
LEFT_POSE_IDXS0 = np.array([502, 504, 506, 508, 510])
RIGHT_POSE_IDXS0 = np.array([503, 505, 507, 509, 511])
LANDMARK_IDXS_LEFT_DOMINANT0 = np.concatenate((LIPS_IDXS0, LEFT_HAND_IDXS0, LEFT_POSE_IDXS0))
LANDMARK_IDXS_RIGHT_DOMINANT0 = np.concatenate((LIPS_IDXS0, RIGHT_HAND_IDXS0, RIGHT_POSE_IDXS0))
HAND_IDXS0 = np.concatenate((LEFT_HAND_IDXS0, RIGHT_HAND_IDXS0), axis=0)
N_COLS = LANDMARK_IDXS_LEFT_DOMINANT0.size

# Landmark indices in processed data
LIPS_IDXS = np.argwhere(np.isin(LANDMARK_IDXS_LEFT_DOMINANT0, LIPS_IDXS0)).squeeze()
LEFT_HAND_IDXS = np.argwhere(np.isin(LANDMARK_IDXS_LEFT_DOMINANT0, LEFT_HAND_IDXS0)).squeeze()
RIGHT_HAND_IDXS = np.argwhere(np.isin(LANDMARK_IDXS_LEFT_DOMINANT0, RIGHT_HAND_IDXS0)).squeeze()
HAND_IDXS = np.argwhere(np.isin(LANDMARK_IDXS_LEFT_DOMINANT0, HAND_IDXS0)).squeeze()
POSE_IDXS = np.argwhere(np.isin(LANDMARK_IDXS_LEFT_DOMINANT0, LEFT_POSE_IDXS0)).squeeze()

LIPS_START = 0
LEFT_HAND_START = LIPS_IDXS.size
RIGHT_HAND_START = LEFT_HAND_START + LEFT_HAND_IDXS.size
POSE_START = RIGHT_HAND_START + RIGHT_HAND_IDXS.size 

In [6]:
train_df = pd.read_csv('/kaggle/input/asl-signs/train.csv')
train_df['sign_ord'] = train_df['sign'].astype('category').cat.codes
ORD2SIGN = train_df[['sign_ord', 'sign']].set_index('sign_ord').squeeze().to_dict()

del train_df

STATS_PATH = '/kaggle/input/datasets/idowuadamo/isolatedasl-data-stats'

LIPS_MEAN = np.load(f'{STATS_PATH}/LIPS_MEAN.npy')
LIPS_STD = np.load(f'{STATS_PATH}/LIPS_STD.npy')
LEFT_HANDS_MEAN = np.load(f'{STATS_PATH}/LEFT_HANDS_MEAN.npy')
LEFT_HANDS_STD = np.load(f'{STATS_PATH}/LEFT_HANDS_STD.npy')
POSE_MEAN = np.load(f'{STATS_PATH}/POSE_MEAN.npy')
POSE_STD = np.load(f'{STATS_PATH}/POSE_STD.npy')

In [7]:
class PreprocessLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(PreprocessLayer, self).__init__()
        normalisation_correction = tf.constant([
                    # Add 0.50 to left hand (original right hand) and substract 0.50 of right hand (original left hand)
                    [0] * len(LIPS_IDXS) + [0.50] * len(LEFT_HAND_IDXS) + [0.50] * len(POSE_IDXS),
                    # Y coordinates stay intact
                    [0] * len(LANDMARK_IDXS_LEFT_DOMINANT0),
                    # Z coordinates stay intact
                    [0] * len(LANDMARK_IDXS_LEFT_DOMINANT0),
                ],
                dtype=tf.float32,
            )
        self.normalisation_correction = tf.transpose(normalisation_correction, [1,0])
        
    def pad_edge(self, t, repeats, side):
        if side == 'LEFT':
            return tf.concat((tf.repeat(t[:1], repeats=repeats, axis=0), t), axis=0)
        elif side == 'RIGHT':
            return tf.concat((t, tf.repeat(t[-1:], repeats=repeats, axis=0)), axis=0)
    
    @tf.function(
        input_signature=(tf.TensorSpec(shape=[None,N_ROWS,N_DIMS], dtype=tf.float32),),
    )
    def call(self, data0):
        # Number of Frames in Video
        N_FRAMES0 = tf.shape(data0)[0]
        
        # Find dominant hand by comparing summed absolute coordinates
        left_hand_sum = tf.math.reduce_sum(tf.where(tf.math.is_nan(tf.gather(data0, LEFT_HAND_IDXS0, axis=1)), 0, 1))
        right_hand_sum = tf.math.reduce_sum(tf.where(tf.math.is_nan(tf.gather(data0, RIGHT_HAND_IDXS0, axis=1)), 0, 1))
        left_dominant = left_hand_sum >= right_hand_sum
        
        # Count non-NaN Hand values in each frame for the dominant hand
        if left_dominant:
            frames_hands_non_nan_sum = tf.math.reduce_sum(
                    tf.where(tf.math.is_nan(tf.gather(data0, LEFT_HAND_IDXS0, axis=1)), 0, 1),
                    axis=[1, 2],
                )
        else:
            frames_hands_non_nan_sum = tf.math.reduce_sum(
                    tf.where(tf.math.is_nan(tf.gather(data0, RIGHT_HAND_IDXS0, axis=1)), 0, 1),
                    axis=[1, 2],
                )
        
        # Find frames indices with coordinates of the dominant hand
        non_empty_frames_idxs = tf.where(frames_hands_non_nan_sum > 0)
        non_empty_frames_idxs = tf.squeeze(non_empty_frames_idxs, axis=1)
        # Filter frames
        data = tf.gather(data0, non_empty_frames_idxs, axis=0)
        
        # Cast Indices in float32 to be compatible with Tensorflow Lite
        non_empty_frames_idxs = tf.cast(non_empty_frames_idxs, tf.float32)
        # Normalize to start with 0
        non_empty_frames_idxs -= tf.reduce_min(non_empty_frames_idxs)
        
        # Number of Frames in Filtered Video
        N_FRAMES = tf.shape(data)[0]
        
        # Gather Relevant Landmark Columns
        if left_dominant:
            data = tf.gather(data, LANDMARK_IDXS_LEFT_DOMINANT0, axis=1)
        else:
            data = tf.gather(data, LANDMARK_IDXS_RIGHT_DOMINANT0, axis=1)
            data = (
                    self.normalisation_correction + (
                        (data - self.normalisation_correction) * tf.where(self.normalisation_correction != 0, -1.0, 1.0))
                )
        
        # Video fits in INPUT_SIZE
        if N_FRAMES < INPUT_SIZE:
            # Pad With -1 to indicate padding
            non_empty_frames_idxs = tf.pad(non_empty_frames_idxs, [[0, INPUT_SIZE-N_FRAMES]], constant_values=-1)
            # Pad Data With Zeros
            data = tf.pad(data, [[0, INPUT_SIZE-N_FRAMES], [0,0], [0,0]], constant_values=0)
            # Fill NaN Values With 0
            data = tf.where(tf.math.is_nan(data), 0.0, data)
            return data, non_empty_frames_idxs
        # Video needs to be downsampled to INPUT_SIZE
        else:
            # Repeat
            if N_FRAMES < INPUT_SIZE**2:
                repeats = tf.math.floordiv(INPUT_SIZE * INPUT_SIZE, N_FRAMES0)
                data = tf.repeat(data, repeats=repeats, axis=0)
                non_empty_frames_idxs = tf.repeat(non_empty_frames_idxs, repeats=repeats, axis=0)

            # Pad To Multiple Of Input Size
            pool_size = tf.math.floordiv(len(data), INPUT_SIZE)
            if tf.math.mod(len(data), INPUT_SIZE) > 0:
                pool_size += 1

            if pool_size == 1:
                pad_size = (pool_size * INPUT_SIZE) - len(data)
            else:
                pad_size = (pool_size * INPUT_SIZE) % len(data)

            # Pad Start/End with Start/End value
            pad_left = tf.math.floordiv(pad_size, 2) + tf.math.floordiv(INPUT_SIZE, 2)
            pad_right = tf.math.floordiv(pad_size, 2) + tf.math.floordiv(INPUT_SIZE, 2)
            if tf.math.mod(pad_size, 2) > 0:
                pad_right += 1

            # Pad By Concatenating Left/Right Edge Values
            data = self.pad_edge(data, pad_left, 'LEFT')
            data = self.pad_edge(data, pad_right, 'RIGHT')

            # Pad Non Empty Frame Indices
            non_empty_frames_idxs = self.pad_edge(non_empty_frames_idxs, pad_left, 'LEFT')
            non_empty_frames_idxs = self.pad_edge(non_empty_frames_idxs, pad_right, 'RIGHT')

            # Reshape to Mean Pool
            data = tf.reshape(data, [INPUT_SIZE, -1, N_COLS, N_DIMS])
            non_empty_frames_idxs = tf.reshape(non_empty_frames_idxs, [INPUT_SIZE, -1])

            # Mean Pool
            data = tf.experimental.numpy.nanmean(data, axis=1)
            non_empty_frames_idxs = tf.experimental.numpy.nanmean(non_empty_frames_idxs, axis=1)

            # Fill NaN Values With 0
            data = tf.where(tf.math.is_nan(data), 0.0, data)
            
            return data, non_empty_frames_idxs
    
preprocess_layer = PreprocessLayer() 

I0000 00:00:1770953484.231961     358 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


# Model Classes (Transformers and Embeddings)

In [8]:
"""
    face: 0:468
    left_hand: 468:489
    pose: 489:522
    right_hand: 522:544
        
"""
def get_data(file_path):
    # Load Raw Data
    data = load_relevant_data_subset(file_path)
    # Process Data Using Tensorflow
    data = preprocess_layer(data)
    
    return data 

In [9]:
def scaled_dot_product(q,k,v, softmax, attention_mask):
    #calculates Q . K(transpose)
    qkt = tf.matmul(q,k,transpose_b=True)
    # calculates scaling factor
    dk = tf.math.sqrt(tf.cast(q.shape[-1],dtype=tf.float32))
    scaled_qkt = qkt/dk
    softmax = softmax(scaled_qkt, mask=attention_mask)
    
    z = tf.matmul(softmax,v)
    #shape: (m,Tx,depth), same shape as q,k,v
    return z

class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self,d_model,num_of_heads):
        super(MultiHeadAttention,self).__init__()
        self.d_model = d_model
        self.num_of_heads = num_of_heads
        self.depth = d_model//num_of_heads
        self.wq = [tf.keras.layers.Dense(self.depth) for i in range(num_of_heads)]
        self.wk = [tf.keras.layers.Dense(self.depth) for i in range(num_of_heads)]
        self.wv = [tf.keras.layers.Dense(self.depth) for i in range(num_of_heads)]
        self.wo = tf.keras.layers.Dense(d_model)
        self.softmax = tf.keras.layers.Softmax()
        
    def call(self,x, attention_mask):
        
        multi_attn = []
        for i in range(self.num_of_heads):
            Q = self.wq[i](x)
            K = self.wk[i](x)
            V = self.wv[i](x)
            multi_attn.append(scaled_dot_product(Q,K,V, self.softmax, attention_mask))
            
        multi_head = tf.concat(multi_attn,axis=-1)
        multi_head_attention = self.wo(multi_head)
        return multi_head_attention

class Transformer(tf.keras.Model):
    def __init__(self, num_blocks):
        super(Transformer, self).__init__(name='transformer')
        self.num_blocks = num_blocks
    
    def build(self, input_shape):
        self.ln_1s = []
        self.mhas = []
        self.ln_2s = []
        self.mlps = []
        # Make Transformer Blocks
        for i in range(self.num_blocks):
            # Multi-Head Attention
            self.mhas.append(MultiHeadAttention(UNITS, 8))
            # Multi Layer Perception
            self.mlps.append(tf.keras.Sequential([
                tf.keras.layers.Dense(UNITS * MLP_RATIO, activation=GELU, kernel_initializer=INIT_GLOROT_UNIFORM),
                tf.keras.layers.Dropout(MLP_DROPOUT_RATIO),
                tf.keras.layers.Dense(UNITS, kernel_initializer=INIT_HE_UNIFORM),
            ]))
        
    def call(self, x, attention_mask):
        # Iterate input over transformer blocks
        for mha, mlp in zip(self.mhas, self.mlps):
            x = x + mha(x, attention_mask)
            x = x + mlp(x)
    
        return x

class LandmarkEmbedding(tf.keras.Model):
    def __init__(self, units, name):
        super(LandmarkEmbedding, self).__init__(name=f'{name}_embedding')
        self.units = units
        
    def build(self, input_shape):
        # Embedding for missing landmark in frame, initialized with zeros
        self.empty_embedding = self.add_weight(
            name=f'{self.name}_empty_embedding',
            shape=[self.units],
            initializer=INIT_ZEROS,
        )
        # Embedding
        self.dense = tf.keras.Sequential([
            tf.keras.layers.Dense(self.units, name=f'{self.name}_dense_1', use_bias=False, kernel_initializer=INIT_GLOROT_UNIFORM),
            tf.keras.layers.Activation(GELU),
            tf.keras.layers.Dense(self.units, name=f'{self.name}_dense_2', use_bias=False, kernel_initializer=INIT_HE_UNIFORM),
        ], name=f'{self.name}_dense')

    def call(self, x):
        return tf.where(
                # Checks whether landmark is missing in frame
                tf.reduce_sum(x, axis=2, keepdims=True) == 0,
                # If so, the empty embedding is used
                self.empty_embedding,
                # Otherwise the landmark data is embedded
                self.dense(x),
            )

class Embedding(tf.keras.Model):
    def __init__(self):
        super(Embedding, self).__init__()
        
    def get_diffs(self, l):
        S = l.shape[2]
        other = tf.expand_dims(l, 3)
        other = tf.repeat(other, S, axis=3)
        other = tf.transpose(other, [0,1,3,2])
        diffs = tf.expand_dims(l, 3) - other
        diffs = tf.reshape(diffs, [-1, INPUT_SIZE, S*S])
        return diffs

    def build(self, input_shape):
        # Positional Embedding, initialized with zeros
        self.positional_embedding = tf.keras.layers.Embedding(INPUT_SIZE+1, UNITS, embeddings_initializer=INIT_ZEROS)
        # Embedding layer for Landmarks
        self.lips_embedding = LandmarkEmbedding(LIPS_UNITS, 'lips')
        self.left_hand_embedding = LandmarkEmbedding(HANDS_UNITS, 'left_hand')
        self.pose_embedding = LandmarkEmbedding(POSE_UNITS, 'pose')
        # Landmark Weights
        self.landmark_weights = tf.Variable(tf.zeros([3], dtype=tf.float32), name='landmark_weights')
        # Fully Connected Layers for combined landmarks
        self.fc = tf.keras.Sequential([
            tf.keras.layers.Dense(UNITS, name='fully_connected_1', use_bias=False, kernel_initializer=INIT_GLOROT_UNIFORM),
            tf.keras.layers.Activation(GELU),
            tf.keras.layers.Dense(UNITS, name='fully_connected_2', use_bias=False, kernel_initializer=INIT_HE_UNIFORM),
        ], name='fc')


    def call(self, lips0, left_hand0, pose0, non_empty_frame_idxs, training=False):
        # Lips
        lips_embedding = self.lips_embedding(lips0)
        # Left Hand
        left_hand_embedding = self.left_hand_embedding(left_hand0)
        # Pose
        pose_embedding = self.pose_embedding(pose0)
        # Merge Embeddings of all landmarks with mean pooling
        x = tf.stack((
            lips_embedding, left_hand_embedding, pose_embedding,
        ), axis=3)
        x = x * tf.nn.softmax(self.landmark_weights)
        x = tf.reduce_sum(x, axis=3)
        # Fully Connected Layers
        x = self.fc(x)
        # Add Positional Embedding
        max_frame_idxs = tf.clip_by_value(
                tf.reduce_max(non_empty_frame_idxs, axis=1, keepdims=True),
                1,
                np.PINF,
            )
        normalised_non_empty_frame_idxs = tf.where(
            tf.math.equal(non_empty_frame_idxs, -1.0),
            INPUT_SIZE,
            tf.cast(
                non_empty_frame_idxs / max_frame_idxs * INPUT_SIZE,
                tf.int32,
            ),
        )
        x = x + self.positional_embedding(normalised_non_empty_frame_idxs)
        
        return x

# Model Loading

In [10]:
def scce_with_ls(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    y_true = tf.one_hot(y_true, NUM_CLASSES, axis=1) 
    return tf.keras.losses.categorical_crossentropy(y_true, y_pred, label_smoothing=0.25) 

def get_model():
    # Inputs
    frames = tf.keras.layers.Input([INPUT_SIZE, N_COLS, N_DIMS], dtype=tf.float32, name='frames')
    non_empty_frame_idxs = tf.keras.layers.Input([INPUT_SIZE], dtype=tf.float32, name='non_empty_frame_idxs')

    # Padding Mask - WRAPPED IN LAMBDA LAYERS
    mask0 = tf.keras.layers.Lambda(lambda x: tf.cast(tf.math.not_equal(x, -1), tf.float32), name='mask0')(non_empty_frame_idxs)
    mask0_expanded = tf.keras.layers.Lambda(lambda x: tf.expand_dims(x, axis=2), name='mask0_expanded')(mask0)

    def create_mask(x):
        # x[0] is mask0_expanded, x[1] is mask0
        mask = tf.where(
            (tf.random.uniform(tf.shape(x[0])) > 0.25) & tf.math.not_equal(x[0], 0.0),
            1.0,
            0.0,
        )
        # Correct Samples Which are all masked now...
        mask = tf.where(
            tf.math.equal(tf.reduce_sum(mask, axis=[1,2], keepdims=True), 0.0),
            x[0], # use mask0_expanded
            mask,
        )
        return mask
    
    mask = tf.keras.layers.Lambda(create_mask, name='create_mask')([mask0_expanded, mask0_expanded]) 

    # Slicing the XY coordinates
    x = tf.keras.layers.Lambda(lambda t: tf.slice(t, [0,0,0,0], [-1,INPUT_SIZE, N_COLS, 2]), name='slice_xy')(frames)
    
    # LIPS 
    lips = tf.keras.layers.Lambda(lambda t: tf.slice(t, [0,0,LIPS_START,0], [-1,INPUT_SIZE, 40, 2]), name='slice_lips')(x)
    lips = tf.keras.layers.Lambda(
        lambda t: tf.where(
            tf.math.equal(t, 0.0), 0.0, (t - LIPS_MEAN) / LIPS_STD
        ), name='normalize_lips')(lips)
    lips = tf.keras.layers.Reshape((INPUT_SIZE, 40*2), name='reshape_lips')(lips)

    # LEFT HAND
    left_hand = tf.keras.layers.Lambda(lambda t: tf.slice(t, [0,0,40,0], [-1,INPUT_SIZE, 21, 2]), name='slice_left_hand')(x)
    left_hand = tf.keras.layers.Lambda(
        lambda t: tf.where(
            tf.math.equal(t, 0.0), 0.0, (t - LEFT_HANDS_MEAN) / LEFT_HANDS_STD
        ), name='normalize_left_hand')(left_hand)
    left_hand = tf.keras.layers.Reshape((INPUT_SIZE, 21*2), name='reshape_left_hand')(left_hand)

    # POSE
    pose = tf.keras.layers.Lambda(lambda t: tf.slice(t, [0,0,61,0], [-1,INPUT_SIZE, 5, 2]), name='slice_pose')(x)
    pose = tf.keras.layers.Lambda(
        lambda t: tf.where(
            tf.math.equal(t, 0.0), 0.0, (t - POSE_MEAN) / POSE_STD
        ), name='normalize_pose')(pose)
    pose = tf.keras.layers.Reshape((INPUT_SIZE, 5*2), name='reshape_pose')(pose)
    
    # Embedding
    x = Embedding()(lips, left_hand, pose, non_empty_frame_idxs)
    
    # Encoder Transformer Blocks
    x = Transformer(NUM_BLOCKS)(x, mask)
    
    # Pooling
    def masked_pooling(tensors):
        x_tensor, mask_tensor = tensors
        return tf.reduce_sum(x_tensor * mask_tensor, axis=1) / tf.reduce_sum(mask_tensor, axis=1)
        
    x = tf.keras.layers.Lambda(masked_pooling, name='masked_pooling')([x, mask])
    
    # Classifier Dropout
    x = tf.keras.layers.Dropout(CLASSIFIER_DROPOUT_RATIO)(x)
    # Classification Layer
    x = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', kernel_initializer=INIT_GLOROT_UNIFORM)(x)
    
    outputs = x
    
    # Create Tensorflow Model
    model = tf.keras.models.Model(inputs=[frames, non_empty_frame_idxs], outputs=outputs)
    
    # Compile (Optional for inference, but keeps loading consistent)
    loss = scce_with_ls
    optimizer = tf.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-5, clipnorm=1.0)
    metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name='acc')]
    model.compile(loss=loss, optimizer=optimizer, metrics=metrics)
    
    return model

# Initialize and Load
model = get_model()

# UPDATE PATH TO YOUR SAVED WEIGHTS
model_weights_path = '/kaggle/input/models/abdulsamadibrahim/model-old-1/tensorflow2/default/1/model.weights.h5'
#model_weights_path = '/kaggle/input/original-model/tensorflow2/default/1/model.weights.h5' 
if os.path.exists(model_weights_path):
    model.load_weights(model_weights_path)
    print("Model weights loaded successfully.")
else:
    print(f"Error: Model weights not found at {model_weights_path}")

Model weights loaded successfully.


# Mediapipe and Frame Extraction

In [11]:
# Setup for MediaPipe Holistic
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) 

# Extract Landmarks From Frame
def extract_landmarks(image):
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = holistic.process(image_rgb)
    landmarks = []
    # Face
    if results.face_landmarks:
        for lm in results.face_landmarks.landmark:
            landmarks.append([lm.x, lm.y, lm.z if hasattr(lm, 'z') else 0])
    else:
        landmarks.extend([[0,0,0]] * 468)
    # Left hand
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            landmarks.append([lm.x, lm.y, lm.z if hasattr(lm, 'z') else 0])
    else:
        landmarks.extend([[0,0,0]] * 21)
    # Pose
    if results.pose_landmarks:
        for lm in results.pose_landmarks.landmark:
            landmarks.append([lm.x, lm.y, lm.z if hasattr(lm, 'z') else 0])
    else:
        landmarks.extend([[0,0,0]] * 33)
    # Right hand
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            landmarks.append([lm.x, lm.y, lm.z if hasattr(lm, 'z') else 0])
    else:
        landmarks.extend([[0,0,0]] * 21)
    landmarks = landmarks[:543]
    return np.array(landmarks, dtype=np.float32)

# LLM Setup

In [12]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(token=secret_value_0, new_session=False) 

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [13]:
# Text Generation Model Setup
llm = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct", device_map="auto")

def refine_to_sentence_fewshot(sign_sequence):
    """
    Convert ASL sign sequence to English using multi-shot prompting with strict output formatting.
    """
    if not sign_sequence:
        return "Signs Detected: None\nSentence: No signs detected."
    
    llm_input = ' '.join(sign_sequence)
    
    prompt = (
        "You are a strictly formatted ASL-to-English translation engine. "
        "Your task is to convert American Sign Language (ASL) glosses into a coherent English sentence.\n\n"
        
        "RULES:\n"
        "1. Output exactly two lines: 'Signs Detected:' followed by the input, and 'Sentence:' followed by the translation.\n"
        "2. Do not add introductions, explanations, or extra text.\n"
        "3. If the input is random noise, incoherent words, or insufficient to form a thought, output '[Unintelligible]'.\n\n"
        
        "EXAMPLES:\n"
        "Input: you name what you\n"
        "Signs Detected: you name what you\n"
        "Sentence: What is your name?\n\n"
        
        "Input: school go morning i\n"
        "Signs Detected: school go morning i\n"
        "Sentence: I go to school in the morning.\n\n"
        
        "Input: purple monkey dishwasher\n"
        "Signs Detected: purple monkey dishwasher\n"
        "Sentence: [Unintelligible]\n\n"
        
        "Input: please help me\n"
        "Signs Detected: please help me\n"
        "Sentence: Please help me.\n\n"
        
        f"Input: {llm_input}\n"
    )
    
    try:
        resp = llm(prompt, max_new_tokens=100, do_sample=False)[0]['generated_text']
        
        # PARSING LOGIC:
        if "Input: " + llm_input in resp:
            output_section = resp.split(f"Input: {llm_input}")[-1].strip()
        else:
            output_section = resp.strip()

        # Clean up any trailing examples
        final_output = output_section.split("Input:")[0].strip()
        return final_output

    except Exception as e:
        return f"Signs Detected: {llm_input}\nSentence: Error processing translation."

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [14]:
# Text to Speech
def text_to_audio(text, audio_path='output.mp3'):
    # Extract just the sentence part for TTS if it's in the structured format
    if "Sentence:" in text:
        text_to_speak = text.split("Sentence:")[-1].strip()
    else:
        text_to_speak = text
        
    if text_to_speak == "[Unintelligible]":
        return None
        
    tts = gTTS(text_to_speak, lang='en')
    tts.save(audio_path)
    return audio_path 

# Video and Live Frame Processing
def process_uploaded_video(video_path):
    if video_path is None:
        return "No video uploaded.", None
    cap = cv2.VideoCapture(video_path)
    frames_landmarks = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        landmarks = extract_landmarks(frame)
        frames_landmarks.append(landmarks)
    cap.release()
    
    if not frames_landmarks:
        return "No frames detected.", None
    
    data = np.array(frames_landmarks)
    window_size = INPUT_SIZE
    step = 16
    predictions = []
    
    # Process video in chunks
    for start in range(0, max(1, len(data) - window_size + 1), step):
        window_data = data[start:start+window_size]
        # Pad last chunk if needed
        if len(window_data) < window_size:
            pad = np.zeros((window_size - len(window_data), 543, 3), dtype=np.float32)
            window_data = np.concatenate((window_data, pad), axis=0)
            
        processed_data, non_empty_idxs = preprocess_layer(window_data)
        processed_data = np.expand_dims(processed_data, 0)
        non_empty_idxs = np.expand_dims(non_empty_idxs, 0)
        
        pred = model.predict({'frames': processed_data, 'non_empty_frame_idxs': non_empty_idxs}, verbose=0)[0].argmax()
        sign = ORD2SIGN.get(pred, 'unknown')
        predictions.append(sign)
        
    # Remove consecutive duplicates and 'unknown'
    unique_seq = []
    for s in predictions:
        if not unique_seq or (s != unique_seq[-1] and s != 'unknown'):
            unique_seq.append(s)
            
    sentence = refine_to_sentence_fewshot(unique_seq)
    audio_path = text_to_audio(sentence)
    return sentence, audio_path

# Globals for Live Stream
frame_buffer = []
sign_sequence = []

def process_live_frame(image):
    global frame_buffer, sign_sequence
    if image is None:
        return "No frame.", None
        
    landmarks = extract_landmarks(image)
    frame_buffer.append(landmarks)
    
    # Wait until buffer fills
    if len(frame_buffer) >= INPUT_SIZE:
        data = np.array(frame_buffer[-INPUT_SIZE:])
        processed_data, non_empty_idxs = preprocess_layer(data)
        processed_data = np.expand_dims(processed_data, 0)
        non_empty_idxs = np.expand_dims(non_empty_idxs, 0)
        
        pred = model.predict({'frames': processed_data, 'non_empty_frame_idxs': non_empty_idxs}, verbose=0)[0].argmax()
        sign = ORD2SIGN.get(pred, 'unknown')
        
        if sign != 'unknown' and (not sign_sequence or sign != sign_sequence[-1]):
            sign_sequence.append(sign)
        
        # Sliding window: keep last 48 frames (overlap)
        frame_buffer[:] = frame_buffer[16:] 
        
    # Trigger Translation every ~10 signs or manually? 
    # For now, we return the current sequence.
    current_status = ' '.join(sign_sequence) if sign_sequence else "Detecting..."
    
    # Auto-translate if sequence gets long (optional logic)
    if len(sign_sequence) > 0 and len(sign_sequence) % 10 == 0:
         # Note: In a real app, you might want a specific 'End' gesture to trigger this
         pass
         
    return current_status, None

def trigger_translation():
    global sign_sequence
    sentence = refine_to_sentence_fewshot(sign_sequence)
    audio_path = text_to_audio(sentence)
    # Optional: Clear sequence after translation
    # sign_sequence = [] 
    return sentence, audio_path

def clear_buffer():
    global frame_buffer, sign_sequence
    frame_buffer = []
    sign_sequence = []
    return "Cleared.", None

In [15]:
with gr.Blocks() as demo:
    gr.Markdown("# Sign Language to Text and Audio")
    with gr.Tabs():
        # TAB 1: Video Upload
        with gr.Tab("Upload Video"):
            video_input = gr.Video(label="Upload Sign Language Video")
            upload_output_text = gr.Text(label="Translated Sentence")
            upload_output_audio = gr.Audio(label="Audio Output")
            video_input.change(
                fn=process_uploaded_video,
                inputs=video_input,
                outputs=[upload_output_text, upload_output_audio]
            )
            
        # TAB 2: Webcam
        with gr.Tab("Live Webcam"):
            with gr.Row():
                webcam_input = gr.Image(sources=["webcam"], streaming=True)
                with gr.Column():
                    live_status_text = gr.Text(label="Detected Glosses (Live)")
                    live_translated_text = gr.Text(label="Final Translation")
                    live_output_audio = gr.Audio(label="Speech")
            
            with gr.Row():
                translate_btn = gr.Button("Translate Now", variant="primary")
                clear_btn = gr.Button("Clear Buffer")

            # Stream frames
            webcam_input.stream(
                fn=process_live_frame,
                inputs=webcam_input,
                outputs=[live_status_text, live_output_audio],
                stream_every=0.1,
                time_limit=60
            )
            
            # Buttons
            translate_btn.click(
                fn=trigger_translation,
                inputs=None,
                outputs=[live_translated_text, live_output_audio]
            )
            
            clear_btn.click(
                fn=clear_buffer,
                inputs=None,
                outputs=[live_status_text, live_output_audio]
            )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://b159519c245a7d8466.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


I0000 00:00:1770954182.460323     445 service.cc:148] XLA service 0x7d5e312385a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1770954182.460366     445 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
E0000 00:00:1770954182.626277     445 cuda_dnn.cc:522] Loaded runtime CuDNN library: 9.1.0 but source was compiled with: 9.3.0.  CuDNN library needs to have matching major version and equal or higher minor version. If using a binary install, upgrade your CuDNN library.  If building from sources, make sure the library loaded at runtime is compatible with the version specified during compile configuration.
E0000 00:00:1770954182.686406     445 cuda_dnn.cc:522] Loaded runtime CuDNN library: 9.1.0 but source was compiled with: 9.3.0.  CuDNN library needs to have matching major version and equal or higher minor version. If using a binary install, upgrade your CuDNN library.  If building from sources